In [99]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [100]:
FILE_PATH = "/Users/chenweihu/Library/CloudStorage/OneDrive-Personal/UChicago/PSI/Green_Data/Electricity Data/CEA_Gas_Reports_Master.xlsx"

# Load the two state × month sheets
gen_raw  = pd.read_excel(FILE_PATH, sheet_name="State × Month (Generation)", header=1)
gas_raw  = pd.read_excel(FILE_PATH, sheet_name="State × Month (Gas Cons)",   header=1)

# Drop the summary 'Total' column and the title row artefact
month_cols = [c for c in gen_raw.columns if c not in ("State", "Total")]

gen = gen_raw[["State"] + month_cols].dropna(subset=["State"])
gas = gas_raw[["State"] + month_cols].dropna(subset=["State"])

# Melt to long format
gen_long = gen.melt(id_vars="State", var_name="Month", value_name="Generation_MU")
gas_long = gas.melt(id_vars="State", var_name="Month", value_name="GasCons_MMSCMD")

# Parse months to datetime for proper x-axis ordering
gen_long["Month_dt"] = pd.to_datetime(gen_long["Month"], format="%b-%y")
gas_long["Month_dt"] = pd.to_datetime(gas_long["Month"], format="%b-%y")

gen_long = gen_long.sort_values(["State", "Month_dt"])
gas_long = gas_long.sort_values(["State", "Month_dt"])

states = sorted(gen["State"].dropna().unique())

# COLOUR PALETTE (13 distinct colours)
palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#aec7e8", "#ffbb78", "#98df8a",
]
color_map = {state: palette[i % len(palette)] for i, state in enumerate(states)}

# FIGURE 1 — Generation by State over Time
fig1 = go.Figure()

for state in states:
    df = gen_long[gen_long["State"] == state]
    fig1.add_trace(go.Scatter(
        x=df["Month_dt"],
        y=df["Generation_MU"],
        mode="lines+markers",
        name=state,
        line=dict(color=color_map[state], width=2),
        marker=dict(size=6),
        hovertemplate=(
            f"<b>{state}</b><br>"
            "Month: %{x|%b %Y}<br>"
            "Generation: %{y:.1f} MU<extra></extra>"
        ),
    ))

fig1.update_layout(
    title=dict(
        text="Gas-Based Power Generation by State<br>"
             "<sup>Jun 2024 – May 2025  |  Million Units (MU)</sup>",
        font=dict(size=18),
    ),
    xaxis=dict(title="Month", tickformat="%b %Y", tickangle=-30, showgrid=True),
    yaxis=dict(title="Generation (MU)", showgrid=True, zeroline=False),
    legend=dict(
        title="State",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#ccc",
        borderwidth=1,
    ),
    hovermode="x unified",
    template="plotly_white",
    width=1100,
    height=620,
)

fig1.show()

# FIGURE 2 — Gas Consumption by State over Time
fig2 = go.Figure()

for state in states:
    df = gas_long[gas_long["State"] == state]
    fig2.add_trace(go.Scatter(
        x=df["Month_dt"],
        y=df["GasCons_MMSCMD"],
        mode="lines+markers",
        name=state,
        line=dict(color=color_map[state], width=2),
        marker=dict(size=6),
        hovertemplate=(
            f"<b>{state}</b><br>"
            "Month: %{x|%b %Y}<br>"
            "Gas Consumed: %{y:.3f} MMSCMD<extra></extra>"
        ),
    ))

fig2.update_layout(
    title=dict(
        text="Gas Consumption by State<br>"
             "<sup>Jun 2024 – May 2025  |  MMSCMD</sup>",
        font=dict(size=18),
    ),
    xaxis=dict(title="Month", tickformat="%b %Y", tickangle=-30, showgrid=True),
    yaxis=dict(title="Gas Consumed (MMSCMD)", showgrid=True, zeroline=False),
    legend=dict(
        title="State",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#ccc",
        borderwidth=1,
    ),
    hovermode="x unified",
    template="plotly_white",
    width=1100,
    height=620,
)

fig2.show()

# FIGURE 3 — Normalised Generation (index = 100 at Jun-24) to compare trends
fig3 = go.Figure()

for state in states:
    df = gen_long[gen_long["State"] == state].copy()
    base = df.loc[df["Month_dt"] == df["Month_dt"].min(), "Generation_MU"].values[0]
    if base and base > 0:
        df["Indexed"] = (df["Generation_MU"] / base) * 100
        fig3.add_trace(go.Scatter(
            x=df["Month_dt"],
            y=df["Indexed"],
            mode="lines+markers",
            name=state,
            line=dict(color=color_map[state], width=2),
            marker=dict(size=6),
            hovertemplate=(
                f"<b>{state}</b><br>"
                "Month: %{x|%b %Y}<br>"
                "Index (Jun-24=100): %{y:.1f}<extra></extra>"
            ),
        ))

fig3.add_hline(y=100, line_dash="dot", line_color="grey", opacity=0.6)

fig3.update_layout(
    title=dict(
        text="Indexed Generation by State (Jun 2024 = 100)<br>"
             "<sup>Normalised to compare relative trends — not absolute scale</sup>",
        font=dict(size=18),
    ),
    xaxis=dict(title="Month", tickformat="%b %Y", tickangle=-30, showgrid=True),
    yaxis=dict(title="Index (Jun-24 = 100)", showgrid=True, zeroline=False),
    legend=dict(
        title="State",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#ccc",
        borderwidth=1,
    ),
    hovermode="x unified",
    template="plotly_white",
    width=1100,
    height=620,
)

fig3.show()

In [93]:
# ── EMISSION FACTORS (kg CO2 per MMSCMD per day → converted to tonnes/month) ─
# Natural gas: ~1.9 kg CO2 per SCM burned
# Domestic APM/KGD6 and RLNG are both methane-dominant but RLNG has slightly
# higher lifecycle emissions due to liquefaction & shipping energy (~5% uplift).
# Sources: IPCC AR6, IEA Gas Combustion EF
EF_DOM_KG_PER_SCM  = 1.90          # kg CO₂e per SCM — domestic pipeline gas
EF_RLNG_KG_PER_SCM = 1.90 * 1.05   # kg CO₂e per SCM — RLNG (5% lifecycle uplift)
SCM_PER_MMSCMD_PER_MONTH = 1e6     # 1 MMSCMD = 1,000,000 SCM/day
DAYS_PER_MONTH = 30                 # approximate

df = pd.read_excel(FILE_PATH, sheet_name="Monthly Summary", header=1)
df = df.dropna(subset=["Month"])
df.columns = df.columns.str.replace("\n", " ").str.strip()

# Rename for convenience
df = df.rename(columns={
    "Total Generation (MU)":          "Generation_MU",
    "Domestic Gas Consumed (MMSCMD)": "Dom_MMSCMD",
    "RLNG Consumed (MMSCMD)":         "RLNG_MMSCMD",
    "Total Gas Consumed (MMSCMD)":    "Total_MMSCMD",
    "% Domestic Share":               "Pct_Dom",
    "% RLNG Share":                   "Pct_RLNG",
})

df["Month_dt"] = pd.to_datetime(df["Month"], format="%b-%y")
df = df.sort_values("Month_dt")

# ── CALCULATE EMISSIONS (thousand tonnes CO₂e) ───────────────────────────────
df["Em_Dom_kt"]  = (df["Dom_MMSCMD"]  * SCM_PER_MMSCMD_PER_MONTH * DAYS_PER_MONTH
                    * EF_DOM_KG_PER_SCM) / 1e6   # kg → kt

df["Em_RLNG_kt"] = (df["RLNG_MMSCMD"] * SCM_PER_MMSCMD_PER_MONTH * DAYS_PER_MONTH
                    * EF_RLNG_KG_PER_SCM) / 1e6

df["Em_Total_kt"] = df["Em_Dom_kt"] + df["Em_RLNG_kt"]

# Emissions intensity: tonnes CO₂e per MU generated
df["Intensity_t_per_MU"] = (df["Em_Total_kt"] * 1000) / df["Generation_MU"]

months      = df["Month_dt"]
month_label = df["Month"].tolist()

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    row_heights=[0.45, 0.30, 0.25],
    vertical_spacing=0.07,
)

# ROW 1 — Stacked bar: Dom + RLNG emissions
fig.add_trace(go.Bar(
    x=months, y=df["Em_Dom_kt"],
    name="Domestic Gas (APM/KGD6)",
    marker_color="#2196F3",
    hovertemplate="<b>Domestic</b><br>%{x|%b %Y}: %{y:.1f} kt CO₂e<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=months, y=df["Em_RLNG_kt"],
    name="RLNG (incl. lifecycle)",
    marker_color="#FF6B35",
    hovertemplate="<b>RLNG</b><br>%{x|%b %Y}: %{y:.1f} kt CO₂e<extra></extra>",
), row=1, col=1)

# Total emissions line overlay
fig.add_trace(go.Scatter(
    x=months, y=df["Em_Total_kt"],
    name="Total Emissions",
    mode="lines+markers",
    line=dict(color="#212121", width=2, dash="dot"),
    marker=dict(size=6, symbol="diamond"),
    hovertemplate="<b>Total</b><br>%{x|%b %Y}: %{y:.1f} kt CO₂e<extra></extra>",
), row=1, col=1)

# ROW 2 — Generation (MU) vs Gas Consumed (MMSCMD)
fig.add_trace(go.Scatter(
    x=months, y=df["Generation_MU"],
    name="Generation (MU)",
    mode="lines+markers",
    line=dict(color="#4CAF50", width=2),
    marker=dict(size=6),
    hovertemplate="%{x|%b %Y}: %{y:.0f} MU<extra></extra>",
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=months, y=df["Total_MMSCMD"],
    name="Gas Consumed (MMSCMD)",
    mode="lines+markers",
    line=dict(color="#9C27B0", width=2, dash="dash"),
    marker=dict(size=6),
    yaxis="y4",
    hovertemplate="%{x|%b %Y}: %{y:.2f} MMSCMD<extra></extra>",
), row=2, col=1)

# ROW 3 — Emissions intensity
fig.add_trace(go.Scatter(
    x=months, y=df["Intensity_t_per_MU"],
    name="Intensity (t CO₂e/MU)",
    mode="lines+markers",
    fill="tozeroy",
    fillcolor="rgba(244,67,54,0.15)",
    line=dict(color="#F44336", width=2),
    marker=dict(size=6),
    hovertemplate="%{x|%b %Y}: %{y:.2f} t CO₂e/MU<extra></extra>",
), row=3, col=1)

# Horizontal reference line — typical CCGT benchmark ~420 t/GWh = 0.42 t/MU
fig.add_hline(
    y=0.42, row=3, col=1,
    line_dash="dot", line_color="grey",
    annotation_text="CCGT benchmark (0.42 t/MU)",
    annotation_position="top right",
    annotation_font_size=11,
)

fig.update_layout(
    barmode="stack",
    title=dict(
        text=(
            "India Gas Power Sector — Emissions Profile for Energy Trading Analysis<br>"
            "<sup>Jun 2024 – May 2025  |  EF: Domestic 1.90 kg CO₂e/SCM · "
            "RLNG 2.00 kg CO₂e/SCM (lifecycle)  |  ~30 days/month assumed</sup>"
        ),
        font=dict(size=17),
        x=0.5,
        y=0.97,
    ),
    template="plotly_white",
    width=1100,
    height=820,
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.01,
        xanchor="right",  x=1,
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#ccc", borderwidth=1,
        font=dict(size=12),
    ),
    hovermode="x unified",
    margin=dict(t=160, b=60, l=70, r=70),
)

fig.update_yaxes(title_text="kt CO₂e",       row=1, col=1, showgrid=True)
fig.update_yaxes(title_text="MU / MMSCMD",   row=2, col=1, showgrid=True)
fig.update_yaxes(title_text="t CO₂e / MU",   row=3, col=1, showgrid=True)
fig.update_xaxes(title_text="Month", tickformat="%b %Y", tickangle=-30, row=3, col=1)

fig.show()

# ── PRINT SUMMARY TABLE ───────────────────────────────────────────────────────
print("\n── Emissions Summary ──────────────────────────────────────────────")
print(f"{'Month':<10} {'Dom (kt)':>10} {'RLNG (kt)':>10} {'Total (kt)':>11} {'Intensity':>12}")
print("-" * 58)
for _, row in df.iterrows():
    print(f"{row['Month']:<10} {row['Em_Dom_kt']:>10.1f} {row['Em_RLNG_kt']:>10.1f} "
          f"{row['Em_Total_kt']:>11.1f} {row['Intensity_t_per_MU']:>10.2f} t/MU")
print("-" * 58)
print(f"{'TOTAL':<10} {df['Em_Dom_kt'].sum():>10.1f} {df['Em_RLNG_kt'].sum():>10.1f} "
      f"{df['Em_Total_kt'].sum():>11.1f}")


── Emissions Summary ──────────────────────────────────────────────
Month        Dom (kt)  RLNG (kt)  Total (kt)    Intensity
----------------------------------------------------------
Jun-24          646.4     1121.0      1767.4     431.88 t/MU
Jul-24          631.6      354.9       986.5     439.91 t/MU
Aug-24          598.5      317.2       915.7     445.08 t/MU
Sep-24          640.1      471.6      1111.7     450.87 t/MU
Oct-24          646.4      218.5       864.8     436.13 t/MU
Nov-24          549.5       55.1       604.5     475.27 t/MU
Dec-24          582.5      144.8       727.4     467.37 t/MU
Jan-25          597.4       85.6       682.9     467.61 t/MU
Feb-25          619.6      169.4       789.0     484.97 t/MU
Mar-25          583.1      184.3       767.4     441.82 t/MU
Apr-25          632.1      714.0      1346.1     426.65 t/MU
May-25          620.0      595.2      1215.3     436.78 t/MU
----------------------------------------------------------
TOTAL          7347.2  

In [92]:
EF_DOM  = 1.90   # kg CO₂e per SCM — domestic gas
EF_RLNG = 2.00   # kg CO₂e per SCM — RLNG (lifecycle)
DAYS = 30

df = pd.read_excel(FILE_PATH, sheet_name="Raw Data", header=1)
df = df.dropna(subset=["State"])
df["Month_dt"] = pd.to_datetime(df["Month"], format="%b-%y")

grp = df.groupby(["State", "Month_dt"]).agg(
    Generation_MU   =("Generation (MU)",        "sum"),
    Dom_MMSCMD      =("Cons Dom Total (MMSCMD)", "sum"),
    RLNG_LT_MMSCMD  =("Cons RLNG LT (MMSCMD)",  "sum"),
    RLNG_Spot_MMSCMD=("Cons RLNG Spot (MMSCMD)", "sum"),
).reset_index()

grp["RLNG_MMSCMD"] = grp["RLNG_LT_MMSCMD"] + grp["RLNG_Spot_MMSCMD"]

# Emissions in kt CO₂e
grp["Em_Dom_kt"]  = grp["Dom_MMSCMD"]  * 1e6 * DAYS * EF_DOM  / 1e6
grp["Em_RLNG_kt"] = grp["RLNG_MMSCMD"] * 1e6 * DAYS * EF_RLNG / 1e6
grp["Em_Total_kt"]= grp["Em_Dom_kt"] + grp["Em_RLNG_kt"]

ann = grp.groupby("State").agg(
    Em_Dom_kt   =("Em_Dom_kt",   "sum"),
    Em_RLNG_kt  =("Em_RLNG_kt",  "sum"),
    Em_Total_kt =("Em_Total_kt", "sum"),
    Generation_MU=("Generation_MU","sum"),
).reset_index()

ann["Intensity"] = (ann["Em_Total_kt"] * 1000) / ann["Generation_MU"].replace(0, float("nan"))
ann["RLNG_Pct"]  = (ann["Em_RLNG_kt"] / ann["Em_Total_kt"] * 100).round(1)
ann = ann.sort_values("Em_Total_kt", ascending=True)  # ascending for horizontal bars

months = sorted(grp["Month_dt"].unique())
states_ordered = ann["State"].tolist()  # bottom→top sorted by total

palette = [
    "#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd",
    "#8c564b","#e377c2","#7f7f7f","#bcbd22","#17becf",
    "#aec7e8","#ffbb78","#98df8a",
]
color_map = {s: palette[i % len(palette)] for i, s in enumerate(states_ordered)}

fig = make_subplots(
    rows=1, cols=3,
    column_widths=[0.42, 0.30, 0.28],
    horizontal_spacing=0.10,
    subplot_titles=(
        "Annual CO₂e Emissions by State & Fuel Source",
        "Monthly Emissions Trend by State",
        "Emissions Intensity vs. RLNG Dependency",
    ),
)

# PANEL 1 — Horizontal stacked bar: Dom + RLNG per state
fig.add_trace(go.Bar(
    y=ann["State"],
    x=ann["Em_Dom_kt"],
    name="Domestic Gas",
    orientation="h",
    marker_color="#2196F3",
    hovertemplate="<b>%{y}</b><br>Domestic: %{x:.0f} kt CO₂e<extra></extra>",
), row=1, col=1)

fig.add_trace(go.Bar(
    y=ann["State"],
    x=ann["Em_RLNG_kt"],
    name="RLNG",
    orientation="h",
    marker_color="#FF6B35",
    hovertemplate="<b>%{y}</b><br>RLNG: %{x:.0f} kt CO₂e<extra></extra>",
), row=1, col=1)

for _, row in ann.iterrows():
    fig.add_annotation(
        y=row["State"], x=row["Em_Total_kt"] + 15,
        text=f"{row['Em_Total_kt']:.0f} kt",
        showarrow=False, font=dict(size=10, color="#333"),
        xref="x", yref="y",
    )

# PANEL 2 — Line chart: monthly emissions per state
for state in reversed(states_ordered):  # top emitters drawn last (on top)
    sub = grp[grp["State"] == state].sort_values("Month_dt")
    fig.add_trace(go.Scatter(
        x=sub["Month_dt"],
        y=sub["Em_Total_kt"],
        name=state,
        mode="lines+markers",
        line=dict(color=color_map[state], width=2),
        marker=dict(size=5),
        legendgroup=state,
        showlegend=False,
        hovertemplate=f"<b>{state}</b><br>%{{x|%b %Y}}: %{{y:.1f}} kt CO₂e<extra></extra>",
    ), row=1, col=2)

# PANEL 3 — Scatter: Intensity vs RLNG%, bubble = total emissions
valid = ann.dropna(subset=["Intensity"])

fig.add_trace(go.Scatter(
    x=valid["RLNG_Pct"],
    y=valid["Intensity"],
    mode="markers+text",
    text=valid["State"].str.title(),
    textposition="top center",
    textfont=dict(size=9),
    marker=dict(
        size=(valid["Em_Total_kt"] / valid["Em_Total_kt"].max() * 50 + 10),
        color=[color_map[s] for s in valid["State"]],
        opacity=0.8,
        line=dict(color="white", width=1),
    ),
    name="States",
    showlegend=False,
    hovertemplate=(
        "<b>%{text}</b><br>"
        "RLNG share: %{x:.1f}%<br>"
        "Intensity: %{y:.1f} t CO₂e/MU<extra></extra>"
    ),
), row=1, col=3)

fig.add_hline(
    y=420, row=1, col=3,
    line_dash="dot", line_color="grey",
    annotation_text="420 t/MU benchmark",
    annotation_position="bottom right",
    annotation_font_size=10,
)

fig.update_layout(
    barmode="stack",
    title=dict(
        text=(
            "India Gas Power Sector — State-Level Emissions Variation  "
            "(Jun 2024 – May 2025)<br>"
            "<sup>EF: Dom 1.90 · RLNG 2.00 kg CO₂e/SCM  |  Bubble size ∝ total annual emissions</sup>"
        ),
        font=dict(size=16),
        x=0.5,
        y=0.97,
    ),
    template="plotly_white",
    width=1400,
    height=680,
    legend=dict(
        title="Fuel Source",
        orientation="h",
        yanchor="bottom", y=1.04,
        xanchor="left", x=0,
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#ccc", borderwidth=1,
    ),
    hovermode="closest",
    margin=dict(t=130, b=60, l=160, r=60),
)

fig.update_xaxes(title_text="kt CO₂e", row=1, col=1, showgrid=True)
fig.update_xaxes(title_text="Month", row=1, col=2, tickformat="%b %y", tickangle=-45)
fig.update_xaxes(title_text="RLNG Share of Emissions (%)", row=1, col=3, showgrid=True)
fig.update_yaxes(title_text="t CO₂e / MU", row=1, col=3, showgrid=True)
fig.update_yaxes(title_text="kt CO₂e / month", row=1, col=2, showgrid=True)
fig.show()